### DUCKDUCKGO ?

In [3]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_ollama import ChatOllama
import ollama
import json

C:\Users\rusha\AppData\Local\Temp\ipykernel_8804\852467463.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults
d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
search = DuckDuckGoSearchResults(max_results=5, region="wt-wt")

In [10]:
def answer_with_search(user_question):
    print("[ USER QUERY ] : ",user_question)
    # Step 1: Ask the model if it knows the answer
    json_type = "{ 'SEARCH' : perfect_search_query_reducing_user_query_into_search_keywords || 'NO_SEARCH' : None }"
    check_prompt = f"""Does answering this question require recent information (news, roadmaps, post-2024 events)?
Question: {user_question}
Reply in JSON with {json_type}."""
    
    check = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': check_prompt}],
                        options={'temperature': 0},
                        format='json')
    # return check
    decision = json.loads(check['message']['content'].strip())
    print("[ DECISION ] : ",decision.get("SEARCH",'') if list(decision.keys())[0]=='SEARCH' else decision.get('NO_SEARCH','nothing') )

    # Step 2: If model says SEARCH, call DuckDuckGo
    if "SEARCH" in decision:
        print("[ SEARCHING ... ]")
        search = DuckDuckGoSearchResults(max_results=5, region="wt-wt", time="y")  # time="y" = past year
        results = search.invoke(user_question)
        # return results
        # Feed results back to model
        final_prompt = f"""Use these search results to answer the question.
Question: {user_question}
Search Results: {results}
also add the links for reference where the information came from
Answer:"""
        response = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': final_prompt}])
        return results,response['message']['content']
    else:
        # Model knows it, answer directly
        response = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': user_question}])
        return response['message']['content']


In [11]:
user_question="can you tell me about recent lunched tesla car and its features?"
result = ""
raw,ans = answer_with_search(user_question)

[ USER QUERY ] :  can you tell me about recent lunched tesla car and its features?
[ DECISION ] :  recent launched Tesla car features
[ SEARCHING ... ]


In [12]:
raw

"snippet: The new Tesla Roadster has been revealed with 3 distinct configurations: Performance, Thrust, and Rocket. Following its October 15, 2026 world premiere, Tesla's next-generation electric sports car ..., title: New Tesla Roadster Revealed - 3 Versions Including a Flying Rocket, link: https://m.youtube.com/watch?v=c25bVS99j7s, snippet: After 9 years of hype, Tesla unveiled the new Roadster at a joint event with SpaceX, where it drove upside down on a ramp and flew over a row of cars. I go t..., title: Tesla's Roadster Just Launched | These Features Are Insane, link: https://m.youtube.com/watch?v=A2SpqBtHvOs, snippet: Tesla's new Roadster features Performance, Thrust and Rocket variants. Discover US reservation deposits, specs, the 2028 production target and worldwide release status., title: Tesla Roadster 2026 Reveal: Flying Car, Price, Variants and Release ..., link: https://boxofficehype.com/tesla-roadster-2026-reveal-flying-car-price-variants-and-release-date/, snippet: Tesla

In [23]:
from pprint import pprint
print(ans)

Based on the search results, here's an overview of the recent Tesla Roadster and its features:

**Name:** Tesla Roadster

**Release Date:** October 15, 2026

**Variants:** The new Tesla Roadster comes in three distinct configurations:

1. **Performance**: The base model, offering a balance of speed and range.
2. **Thrust**: A higher-performance variant with improved acceleration and range.
3. **Rocket**: The most advanced and extreme variant, featuring SpaceX-inspired thrusters that allow the car to fly.

**Features:**

* Flying capability using SpaceX-inspired thrusters, enabling the Rocket variant to fly over a row of cars.
* Unique stunt showcased during the car's world premiere, where it drove upside down on a ramp and flew over a row of cars.
* Improved performance, acceleration, and range compared to the previous generation.

**Production and Release:**

* 2028 production target
* Worldwide release status: available for reservation deposits

**Additional Resources:**

* For more 

In [13]:
items = []
i=0
for block in raw.split("snippet: ")[1:]:
    i+=1
    print('count',i)
    parts = block.split(", title: ")
    snippet = parts[0]
    title_link = parts[1].split(", link: ")
    title = title_link[0]
    link = title_link[1].strip().rstrip(",")
    items.append({"snippet": snippet, "title": title, "link": link})


count 1
count 2
count 3
count 4


In [17]:
for x in items:
    print(x.get('link'))

https://m.youtube.com/watch?v=c25bVS99j7s
https://m.youtube.com/watch?v=A2SpqBtHvOs
https://boxofficehype.com/tesla-roadster-2026-reveal-flying-car-price-variants-and-release-date/
https://www.usatoday.com/story/cars/news/2026/10/16/tesla-roadster-flying-spacex-thrusters/92279108007/


In [29]:
with open ('search_results.json', 'w') as f:
    json.dump(items, f)